# LLM_RE_Local no Colab

Sobe o `llama-server` (llama.cpp, GPU T4) com o modelo DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic (i1-Q6_K).

**Antes de começar:** Ambiente de execução -> Alterar o tipo de ambiente de execução -> GPU T4. Use uma conta Google secundária.

**Níveis e termos do Colab**
- **N1 (células 1, 2, 3):** chat dentro do notebook, sem túnel. Dentro das regras do Colab.
- **N2 (célula 4):** túnel cloudflared para usar o servidor pelo terminal do PC. Isso **viola o FAQ do plano gratuito** do Colab (contornar a interface do notebook / controle remoto). O risco é seu: use só conta secundária, ligue o túnel só quando for usar e desligue ao terminar.

**Ao terminar:** Ambiente de execução -> **Desconectar e excluir ambiente de execução**.

## Célula 1: GPU e binário llama.cpp (CUDA 12.8)
Confere a GPU e baixa da API do GitHub a primeira release recente que tenha o binário `ubuntu-cuda-12.8-x64` (a release "latest" pode não ter binários), mais o `cudart` correspondente, se existir. Se o `llama-server --version` falhar, use a célula 1b.

In [ ]:
!nvidia-smi
import json, re, subprocess, urllib.request, os, glob

def _get(url):
    req = urllib.request.Request(url, headers={"User-Agent": "llm-local"})
    return urllib.request.urlopen(req, timeout=60)

releases = json.load(_get("https://api.github.com/repos/ggml-org/llama.cpp/releases?per_page=10"))
main = cuda = None
for r in releases:
    for a in r["assets"]:
        if re.match(r"^llama-b\d+-bin-ubuntu-cuda-12\.8-x64\.tar\.gz$", a["name"]):
            main = a
    if main:
        num = re.search(r"llama-(b\d+)", main["name"]).group(1)
        cuda = next((a for a in r["assets"] if a["name"] == f"cudart-llama-{num}-bin-ubuntu-cuda-12.8-x64.tar.gz"), None)
        break
assert main, "Nenhuma release recente com binario ubuntu-cuda-12.8-x64; use a celula 1b."

os.makedirs("/content/llama", exist_ok=True)
for a in filter(None, [main, cuda]):
    print("Baixando", a["name"])
    subprocess.run(["wget", "-q", "-O", "/content/" + a["name"], a["browser_download_url"]], check=True)
    subprocess.run(["tar", "-xzf", "/content/" + a["name"], "-C", "/content/llama"], check=True)

SERVER = glob.glob("/content/llama/**/llama-server", recursive=True)[0]
subprocess.run(["chmod", "+x", SERVER])
LIBDIR = ":".join(sorted({os.path.dirname(p) for p in glob.glob("/content/llama/**/*.so*", recursive=True)}))  # inclui a pasta do cudart
env = dict(os.environ, LD_LIBRARY_PATH=LIBDIR + ":" + os.environ.get("LD_LIBRARY_PATH", ""))
r = subprocess.run([SERVER, "--version"], env=env, capture_output=True, text=True)
print(r.stdout, r.stderr)
if r.returncode != 0:
    print("llama-server nao rodou. Execute a celula 1b (compilar com CUDA para sm_75).")

## Célula 1b (opcional): compilar do código-fonte
Só se a célula 1 falhar. Leva ~10-15 min. Para usar, remova o `#` das linhas de comando e execute. Depois ajuste `SERVER` e `LIBDIR` como indicado no final.

In [ ]:
# !git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
# %cd /content/llama.cpp
# !cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75
# !cmake --build build -j --target llama-server
# SERVER = "/content/llama.cpp/build/bin/llama-server"
# LIBDIR = "/content/llama.cpp/build/bin"

## Célula 2: modelo e servidor
Baixa o modelo i1-Q6_K (~7,4 GB; é baixado de novo a cada sessão), gera uma chave aleatória (`KEY`) e sobe o servidor em segundo plano em `127.0.0.1:8081`, com `--api-key`. O log fica em `/content/server.log`. Espera o `/health` ficar pronto.

In [ ]:
import secrets, subprocess, time, os, urllib.request
from huggingface_hub import hf_hub_download

MODEL = hf_hub_download(
    "mradermacher/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-i1-GGUF",
    "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic.i1-Q6_K.gguf",
    local_dir="/content/models",
)
KEY = secrets.token_urlsafe(24)
env = dict(os.environ, LD_LIBRARY_PATH=LIBDIR + ":" + os.environ.get("LD_LIBRARY_PATH", ""))
server_log = open("/content/server.log", "wb")
server = subprocess.Popen(
    [SERVER, "-m", MODEL, "-ngl", "99", "-c", "16384", "--jinja",
     "--host", "127.0.0.1", "--port", "8081", "--api-key", KEY],
    stdout=server_log, stderr=subprocess.STDOUT, env=env)

for _ in range(180):
    if server.poll() is not None:
        raise RuntimeError("llama-server encerrou; veja /content/server.log")
    try:
        urllib.request.urlopen("http://127.0.0.1:8081/health", timeout=2)
        print("Servidor pronto.")
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("Servidor nao ficou pronto; veja /content/server.log")

## Célula 3 (N1): chat dentro do notebook
Sem túnel, dentro das regras do Colab. Digite `sair` para encerrar o chat.

In [ ]:
import json, requests

history = []
while True:
    msg = input("voce> ").strip()
    if msg == "sair":
        break
    if not msg:
        continue
    history.append({"role": "user", "content": msg})
    resp = requests.post(
        "http://127.0.0.1:8081/v1/chat/completions",
        headers={"Authorization": "Bearer " + KEY},
        json={"messages": history, "stream": True}, stream=True)
    texto = ""
    for line in resp.iter_lines():
        line = line.decode("utf-8")
        if not line.startswith("data:") or line.endswith("[DONE]"):
            continue
        delta = json.loads(line[5:])["choices"][0]["delta"]
        piece = delta.get("content") or ""
        texto += piece
        print(piece, end="", flush=True)
    print()
    history.append({"role": "assistant", "content": texto})

## Célula 4 (N2): túnel para usar pelo terminal do PC
**Atenção:** viola o FAQ do plano gratuito do Colab; risco seu, conta secundária. Baixa o `cloudflared`, abre um quick tunnel (sem conta) para o servidor e imprime a linha para colar no terminal do Windows. A URL muda a cada sessão; a chave protege o acesso.

In [ ]:
import re, subprocess, time

!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared

tunnel_log = open("/content/tunnel.log", "wb")
tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8081", "--no-autoupdate"],
    stdout=tunnel_log, stderr=subprocess.STDOUT)

URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log", errors="ignore").read())
    if m:
        URL = m.group(0)
        break
    time.sleep(1)
assert URL, "URL do tunel nao apareceu; veja /content/tunnel.log"
print("Cole no terminal do Windows:\n")
print(f"llm colab {URL} {KEY}")

## Célula 5: encerrar o túnel
Rode assim que terminar de usar o terminal.

In [ ]:
tunnel.terminate()
print("Tunel encerrado.")

## Célula 6: encerrar o servidor
Depois, feche tudo: Ambiente de execução -> **Desconectar e excluir ambiente de execução**.

In [ ]:
server.terminate()
print("Servidor encerrado.")